# Custom Attention Mechanism & SMS Spam Classification
**GPU recommande : Runtime > Change runtime type > T4 GPU**

## Part 1 : Setup & Data Loading

### 1. Install required packages

In [ ]:
!pip install --quiet datasets evaluate transformers[sentencepiece]


### 2. Import necessary modules

In [ ]:
import pandas as pd
from datasets import Dataset
from datasets import load_dataset

print('Imports OK')


### 3. Load & inspect the SMS Spam dataset

In [ ]:
# Chargement du dataset SMS Spam depuis HuggingFace
df = pd.read_parquet('hf://datasets/ucirvine/sms_spam/plain_text/train-00000-of-00001.parquet')

# Conversion en HuggingFace Dataset
hf_dataset = Dataset.from_pandas(df)

# Split : 4000 pour l'entrainement, 1000 pour la validation
train_ds = hf_dataset.select(range(4000))
val_ds   = hf_dataset.select(range(4000, 5000))

# Affichage des premieres lignes
df.head()


## Part 2 : Tokenization Setup

### 1. Initialize the tokenizer

In [ ]:
from transformers import GPT2Tokenizer

model_name = 'gpt2'

# Chargement du tokenizer GPT-2
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# GPT-2 n'a pas de pad token par defaut -> on utilise le token de fin de sequence
tokenizer.pad_token = tokenizer.eos_token

print(f'Tokenizer charge. Vocab size: {len(tokenizer)}')
print(f'Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})')


### 2. Create tokenization function

In [ ]:
def tokenize_fn(examples):
    return tokenizer(
        examples['sms'],          # colonne contenant le texte
        padding='max_length',     # completer jusqu'a max_length
        truncation=True,          # couper si trop long
        max_length=64             # longueur maximale fixee a 64
    )

# Application a chaque dataset
train_tok = train_ds.map(tokenize_fn, batched=True)
val_tok   = val_ds.map(tokenize_fn, batched=True)

print('Tokenisation terminee.')
print(f'Exemple - input_ids: {train_tok[0]["input_ids"][:10]}...')


## Part 3 : Pre-trained Model Setup

### 1. Initialize GPT-2 for sequence classification

In [ ]:
import torch
from transformers import GPT2ForSequenceClassification

model = GPT2ForSequenceClassification.from_pretrained(
    'gpt2',
    num_labels=2,                              # classification binaire : spam vs ham
    pad_token_id=tokenizer.eos_token_id        # necesaire car GPT-2 n'a pas de pad token natif
)

print('Modele GPT-2 charge avec tete de classification (2 classes).')
print(f'Nombre de parametres: {sum(p.numel() for p in model.parameters()):,}')


## Part 4 : Custom Attention Implementation

### 1. Implement the Attention class

In [ ]:
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        # Facteur de mise a l'echelle : evite que les produits scalaires
        # deviennent trop grands quand embed_dim est eleve
        self.scale = embed_dim ** -0.5

    def forward(self, query, key, value, mask=None):
        # Calcul des scores d'attention : similarite entre chaque paire (query, key)
        # query : (batch, seq, dim) | key.transpose(-2,-1) : (batch, dim, seq)
        # scores : (batch, seq, seq)
        scores = torch.matmul(query, key.transpose(-2, -1)) * self.scale

        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))

        # Softmax sur la derniere dimension : les scores somment a 1 par ligne
        attn = F.softmax(scores, dim=-1)

        # Multiplication par les valeurs : agregation ponderee
        return torch.matmul(attn, value), attn

print('Classe Attention definie.')


### 2. Build the attention-based classifier

In [ ]:
class SimpleAttentionClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        # Couche d'embedding : convertit chaque token ID en vecteur dense
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        # Couche d'attention
        self.attn = Attention(embed_dim)
        # Couche de classification finale
        self.fc = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x : (batch, seq_len) -> embed : (batch, seq_len, embed_dim)
        embed = self.embedding(x)
        # Self-attention : query = key = value = embed
        attn_output, _ = self.attn(embed, embed, embed)
        # Pooling : moyenne sur la dimension sequence -> (batch, embed_dim)
        pooled = attn_output.mean(dim=1)
        # Classification -> (batch, num_classes)
        return self.fc(pooled)

print('Classe SimpleAttentionClassifier definie.')


### 3. Prepare data for the custom model

In [ ]:
def preprocess_for_attention(example):
    tokens = tokenizer.encode(
        example['sms'],
        max_length=64,
        truncation=True,
        padding='max_length'
    )
    return {'input_ids': tokens, 'label': example['label']}

train_ds_attn = train_ds.map(preprocess_for_attention)
val_ds_attn   = val_ds.map(preprocess_for_attention)

print('Preprocessing termine.')
print(f'Exemple label: {train_ds_attn[0]["label"]} | ids[:5]: {train_ds_attn[0]["input_ids"][:5]}')


### 4. Create PyTorch DataLoader

In [ ]:
class SMSDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids': torch.tensor(item['input_ids'], dtype=torch.long),
            'label':     torch.tensor(item['label'],     dtype=torch.long)
        }

train_loader = DataLoader(SMSDataset(train_ds_attn), batch_size=32, shuffle=True)
val_loader   = DataLoader(SMSDataset(val_ds_attn),   batch_size=32)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


### 5. Train the custom attention model

In [ ]:
vocab_size  = len(tokenizer)   # taille du vocabulaire GPT-2 (~50 257)
embed_dim   = 64               # dimension des embeddings
num_classes = 2                # binaire : ham (0) vs spam (1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilise : {device}')

attn_model = SimpleAttentionClassifier(vocab_size, embed_dim, num_classes).to(device)

optimizer = torch.optim.Adam(attn_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

# Boucle d'entrainement (1 epoque)
attn_model.train()
for batch in train_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)

    optimizer.zero_grad()          # reinitialiser les gradients
    outputs = attn_model(inputs)   # passe avant
    loss = criterion(outputs, labels)  # calcul de la perte
    loss.backward()                # retropropagation
    optimizer.step()               # mise a jour des poids

print(f'Custom Attention model trained. Sample batch loss: {loss.item():.4f}')


## Part 5 : Metrics & Evaluation

### 1. Define evaluation metrics

In [ ]:
import evaluate
import numpy as np

accuracy  = evaluate.load('accuracy')
precision = evaluate.load('precision')
recall    = evaluate.load('recall')
f1        = evaluate.load('f1')

def compute_metrics(pred):
    logits, labels = pred
    preds = np.argmax(logits, axis=-1)
    return {
        'accuracy':  accuracy.compute( predictions=preds, references=labels)['accuracy'],
        'precision': precision.compute(predictions=preds, references=labels)['precision'],
        'recall':    recall.compute(   predictions=preds, references=labels)['recall'],
        'f1':        f1.compute(       predictions=preds, references=labels)['f1']
    }

print('Metriques chargees : accuracy, precision, recall, f1')


### 2. Evaluate both models

In [ ]:
# ── Evaluation GPT-2 ──────────────────────────────────────────────────────
print('Evaluation GPT-2 Model...')
gpt2_preds  = []
gpt2_labels = []

model.eval()
for ex in val_tok:
    inputs = torch.tensor(ex['input_ids']).unsqueeze(0).to(model.device)
    with torch.no_grad():
        logits = model(inputs).logits
    pred = torch.argmax(logits, dim=-1).cpu().item()
    gpt2_preds.append(pred)
    gpt2_labels.append(ex['label'])

gpt2_metrics = {
    'accuracy':  accuracy.compute( predictions=gpt2_preds, references=gpt2_labels)['accuracy'],
    'precision': precision.compute(predictions=gpt2_preds, references=gpt2_labels)['precision'],
    'recall':    recall.compute(   predictions=gpt2_preds, references=gpt2_labels)['recall'],
    'f1':        f1.compute(       predictions=gpt2_preds, references=gpt2_labels)['f1']
}
print('GPT-2 Metrics:', gpt2_metrics)

# ── Evaluation Custom Attention ────────────────────────────────────────────
print('\nEvaluation Custom Attention Model...')
attn_preds  = []
attn_labels = []

attn_model.eval()
for batch in val_loader:
    inputs = batch['input_ids'].to(device)
    labels = batch['label'].to(device)
    with torch.no_grad():
        outputs = attn_model(inputs)
        preds   = torch.argmax(outputs, dim=1)
    attn_preds.extend(preds.cpu().tolist())
    attn_labels.extend(labels.cpu().tolist())

attn_metrics = {
    'accuracy':  accuracy.compute( predictions=attn_preds, references=attn_labels)['accuracy'],
    'precision': precision.compute(predictions=attn_preds, references=attn_labels)['precision'],
    'recall':    recall.compute(   predictions=attn_preds, references=attn_labels)['recall'],
    'f1':        f1.compute(       predictions=attn_preds, references=attn_labels)['f1']
}
print('Attention Model Metrics:', attn_metrics)


### Tableau comparatif des resultats

In [ ]:
print(f'\n{"="*55}')
print(f'{"Metrique":<15} {"GPT-2":>15} {"Custom Attn":>15}')
print(f'{"="*55}')
for key in ['accuracy', 'precision', 'recall', 'f1']:
    g = gpt2_metrics[key]
    a = attn_metrics[key]
    print(f'{key:<15} {g:>15.4f} {a:>15.4f}')
print(f'{"="*55}')


## Part 6 : Reflection Questions

### 1. Roles de Query, Key et Value

- **Query** : ce que l'on cherche. Chaque position de la sequence pose une question : "A quoi dois-je preter attention ?"
- **Key** : ce que chaque position offre comme etiquette. Le score d'attention est calcule en mesurant la similarite entre la query et chaque key (`score = Q * K^T`).
- **Value** : le contenu reel transmis. Une fois les scores calcules (softmax), on fait une moyenne ponderee des values : les positions les plus pertinentes contribuent le plus a la sortie.

**Analogie** : dans une bibliotheque, la query est votre question, les keys sont les titres des livres, et les values sont le contenu des livres.

### 2. Pourquoi le facteur d'echelle 1/sqrt(d_k) ?

Quand la dimension `d_k` est grande, les produits scalaires Q*K^T deviennent numeriquement tres grands. Cela pousse le softmax vers des regions ou son gradient est quasi nul (saturation), ce qui bloque l'apprentissage (gradient vanishing).
Diviser par `sqrt(d_k)` normalise les scores et maintient les gradients dans une plage exploitable, quelle que soit la dimension des embeddings.

### 3. Self-attention vs RNNs

| Critere | RNN | Self-Attention |
|---|---|---|
| Traitement | Sequentiel (token par token) | Parallele (tous les tokens en meme temps) |
| Dependances longues | Difficile (gradient vanishing) | Directe (toute paire en O(1) saut) |
| Complexite | O(n) en temps, parallelisable difficilement | O(n²) en memoire mais tres parallelisable |
| Entrainement | Plus lent | Beaucoup plus rapide sur GPU |

La self-attention peut connecter deux tokens distants en une seule operation, la ou un RNN doit propager l'information a travers toutes les etapes intermediaires.

### 4. Performance Analysis

**GPT-2** obtient generalement des metriques plus faibles ici car il n'a PAS ete fine-tune sur cette tache — on utilise juste ses poids pre-entraines avec une tete de classification non entrainee. Ses predictions sont donc quasi aleatoires.

**Custom Attention** a ete entraine directement sur les donnees SMS (meme 1 epoque), il apprend donc les patterns du dataset et obtient de meilleures performances sur cette tache precise.

**Trade-offs** :
- GPT-2 : 117M parametres, generaliste, necessite du fine-tuning pour une tache specifique
- Custom : ~3M parametres, leger, entraine from scratch, moins generaliste mais adapte

**Pour ameliorer le modele custom** : ajouter plusieurs epochs, utiliser multi-head attention, ajouter du dropout, utiliser un scheduler de learning rate.